# Iris pipeline — Abhinav Harsh, 23BCS006
CS-8037. Course baseline with AI-assisted implementation and verification. Own live runs; C4 adds a missing-value/class guard. No bonus attempted.

Displayed outputs were captured from this notebook’s real Colab run on 16 November 2026. The temporary UI secret output is intentionally cleared.

In [ ]:
import os,sys,subprocess,json
from pathlib import Path
COLAB = 'COLAB_RELEASE_TAG' in os.environ
ROOT = Path('/content/freshcart-pipelines') if COLAB else Path.cwd()
if not COLAB and ROOT.name == 'iris_project': ROOT=ROOT.parent
ROOT.mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
FILES = {'iris_project/metadata.yaml': 'project_type: standalone\nvariables_dir: .local/mage_data\nremote_variables_dir:\nproject_uuid: 48779e1c6f8e468d8284d658515b79e7\n', 'iris_project/__init__.py': '', 'iris_project/depth_experiment.py': '"""Run the actual Mage CLI for all five required depth values."""\nfrom pathlib import Path\nimport subprocess,sys,json,re\nROOT=Path(__file__).resolve().parents[1]\nOUT=ROOT/\'evidence\';OUT.mkdir(exist_ok=True)\nrows=[]\nfor depth in range(1,6):\n    (ROOT/\'iris_predictions.csv\').unlink(missing_ok=True)\n    cmd=[str(Path(sys.executable).with_name(\'mage\')),\'run\',\'iris_project\',\'iris_classifier\',\'--test\',\'--runtime-vars\',json.dumps({\'max_depth\':depth})]\n    r=subprocess.run(cmd,cwd=ROOT,capture_output=True,text=True,timeout=120)\n    log=r.stdout+r.stderr\n    (OUT/f\'iris-depth-{depth}.txt\').write_text(log)\n    match=re.search(r\'Test accuracy = ([0-9.]+)\',log)\n    rows.append({\'max_depth\':depth,\'accuracy\':float(match.group(1)) if match else None,\'passed\':r.returncode==0,\'csv_created\':(ROOT/\'iris_predictions.csv\').exists()})\n    assert rows[-1][\'csv_created\']==(depth!=1), rows[-1]\nprint(json.dumps(rows,indent=2))\n(OUT/\'iris-depths.json\').write_text(json.dumps(rows,indent=2)+\'\\n\')\nr=subprocess.run([str(Path(sys.executable).with_name(\'mage\')),\'run\',\'iris_project\',\'iris_classifier\',\'--test\'],cwd=ROOT,capture_output=True,text=True,timeout=120)\n(OUT/\'iris-default.txt\').write_text(r.stdout+r.stderr)\nassert r.returncode==0\nprint(\'Restored default depth 3 successfully\')\n', 'iris_project/pipelines/__init__.py': '', 'iris_project/transformers/evaluate_model.py': "import pandas as pd\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score\n\nif 'transformer' not in globals():\n    from mage_ai.data_preparation.decorators import transformer\n\n\n@transformer\ndef evaluate(df, *args, **kwargs):\n    y_true = df['target']\n    y_pred = df['predicted']\n\n    print('Confusion matrix:')\n    print(pd.crosstab(y_true, y_pred, rownames=['actual'], colnames=['predicted']))\n\n    metrics = pd.DataFrame({\n        'metric': ['accuracy', 'precision', 'recall', 'f1_score'],\n        'value': [\n            round(accuracy_score(y_true, y_pred), 3),\n            round(precision_score(y_true, y_pred, average='macro'), 3),\n            round(recall_score(y_true, y_pred, average='macro'), 3),\n            round(f1_score(y_true, y_pred, average='macro'), 3),\n        ]\n    })\n    return metrics\n\nif 'test' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n@test\ndef test_metrics(output, *args):\n    assert set(output['metric']) == {'accuracy','precision','recall','f1_score'}\n    assert output['value'].between(0, 1).all()\n", 'iris_project/transformers/train_model.py': "import joblib\nfrom sklearn.tree import DecisionTreeClassifier\n\n# Import Mage's decorators only if they aren't already loaded\nif 'transformer' not in globals():\n    from mage_ai.data_preparation.decorators import transformer\nif 'test' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n# The 4 measurement columns the model uses to predict the species\nFEATURES = ['sepal length (cm)', 'sepal width (cm)',\n            'petal length (cm)', 'petal width (cm)']\n\n\n# transformer receives prepare_data's table (150 rows with 'split') as 'df'\n# Reads max_depth from Mage's pipeline variables, then trains a decision tree on the train rows\n# Predicts the 30 test rows, prints accuracy, saves the model and returns the predictions\n@transformer\ndef train(df, *args, **kwargs):\n    max_depth = int(kwargs.get('max_depth', 3))\n\n    train_df = df[df['split'] == 'train']\n    test_df = df[df['split'] == 'test'].copy()\n\n    model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)\n    model.fit(train_df[FEATURES], train_df['target'])\n\n    test_df['predicted'] = model.predict(test_df[FEATURES])\n    accuracy = (test_df['predicted'] == test_df['target']).mean()\n    print('max_depth =', max_depth, '| Test accuracy =', round(accuracy, 3))\n\n    joblib.dump(model, 'iris_model.pkl')\n    return test_df\n\n\n# Runs right after train finishes, using its 30 predicted rows as 'output'\n# Recalculates accuracy by comparing 'predicted' with the true 'target'\n# If accuracy is below 80%, the block fails and save_results / evaluate_model never run\n@test\ndef test_accuracy(output, *args):\n    accuracy = (output['predicted'] == output['target']).mean()\n    assert accuracy >= 0.8, 'Model accuracy is below 80%'\n", 'iris_project/transformers/prepare_data.py': "import pandas as pd\nfrom sklearn.model_selection import train_test_split\n\n# Import Mage's decorators only if they aren't already loaded\nif 'transformer' not in globals():\n    from mage_ai.data_preparation.decorators import transformer\nif 'test' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n\n# ransformer marks this as a step that changes data coming from an earlier block\n# 'df' is the 150-row table returned by load_iris, passed in automatically by Mage\n# It splits the data 80/20 and adds a 'split' column labelling each row train or test\n@transformer\ndef transform(df, *args, **kwargs):\n    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)\n\n    train_df = train_df.copy()\n    test_df = test_df.copy()\n    train_df['split'] = 'train'\n    test_df['split'] = 'test'\n\n    result = pd.concat([train_df, test_df])\n    print('Train rows:', len(train_df), '| Test rows:', len(test_df))\n    return result\n\n\n# Runs right after transform finishes, using its returned table as 'output'\n# Checks that the new 'split' column was actually created\n# If it's missing, the block fails and train_model never runs\n@test\ndef test_output(output, *args):\n    assert 'split' in output.columns, 'split column missing'\n", 'iris_project/transformers/__init__.py': '', 'iris_project/transformers/verify_iris.py': '"""C4 extension: validate the input measurements before splitting."""\nif \'transformer\' not in globals():\n    from mage_ai.data_preparation.decorators import transformer\nif \'test\' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n@transformer\ndef verify(df, *args, **kwargs):\n    return df.copy()\n\n@test\ndef test_input_quality(output, *args):\n    assert not output.isna().any().any(), \'Iris data contains missing values\'\n    assert set(output[\'target\']) == {0, 1, 2}, \'Expected exactly three Iris classes\'\n', 'iris_project/data_exporters/__init__.py': '', 'iris_project/data_exporters/save_results.py': "# Import Mage's decorator only if it isn't already loaded\nif 'data_exporter' not in globals():\n    from mage_ai.data_preparation.decorators import data_exporter\n\n\n# data_exporter marks this as a step that sends data out of the pipeline\n# 'df' is the 30 predicted rows returned by train_model, passed in automatically by Mage\n# It saves them to a CSV file and returns nothing, because this is an end of the pipeline\n@data_exporter\ndef export_data(df, *args, **kwargs):\n    df.to_csv('iris_predictions.csv', index=False)\n    print('Saved', len(df), 'predictions to iris_predictions.csv')\n\nif 'test' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n@test\ndef test_saved_predictions(*args):\n    import pandas as pd\n    saved = pd.read_csv('iris_predictions.csv')\n    assert len(saved) == 30 and 'predicted' in saved.columns\n", 'iris_project/data_loaders/__init__.py': '', 'iris_project/data_loaders/load_iris.py': "import pandas as pd\nfrom sklearn.datasets import load_iris\n\n# One-line version for students: “If Mage hasn’t already provided the data_loader\n# and test labels, import them, so the file works both inside and outside Mage.\n\n# Import Mage's decorators only if they aren't already loaded\nif 'data_loader' not in globals():\n    from mage_ai.data_preparation.decorators import data_loader\nif 'test' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n\n# data_loader marks this function as the pipeline's first step: it brings data in\n# It loads the built-in Iris dataset from scikit-learn as a pandas table\n# The returned DataFrame is passed automatically to the next block (prepare_data)\n@data_loader\ndef load_data(*args, **kwargs):\n    iris = load_iris(as_frame=True)\n    df = iris.frame                      # 4 feature columns + 'target'\n    print('Rows loaded:', len(df))\n    return df\n\n\n# test marks this as a quality check that Mage runs right after the block finishes\n# 'output' is whatever load_data returned, so we can verify the data\n# If the table doesn't have exactly 150 rows, the block fails and the pipeline stops\n@test\ndef test_output(output, *args):\n    assert len(output) == 150, 'Iris must have 150 rows'\n", 'iris_project/pipelines/iris_classifier/metadata.yaml': 'blocks:\n- name: load_iris\n  uuid: load_iris\n  type: data_loader\n  language: python\n  upstream_blocks: []\n  downstream_blocks: [verify_iris]\n- name: verify_iris\n  uuid: verify_iris\n  type: transformer\n  language: python\n  upstream_blocks: [load_iris]\n  downstream_blocks: [prepare_data]\n- name: prepare_data\n  uuid: prepare_data\n  type: transformer\n  language: python\n  upstream_blocks: [verify_iris]\n  downstream_blocks: [train_model]\n- name: train_model\n  uuid: train_model\n  type: transformer\n  language: python\n  upstream_blocks: [prepare_data]\n  downstream_blocks: [save_results, evaluate_model]\n- name: save_results\n  uuid: save_results\n  type: data_exporter\n  language: python\n  upstream_blocks: [train_model]\n  downstream_blocks: []\n- name: evaluate_model\n  uuid: evaluate_model\n  type: transformer\n  language: python\n  upstream_blocks: [train_model]\n  downstream_blocks: []\nname: iris_classifier\nuuid: iris_classifier\ntype: python\nvariables:\n  max_depth: 3\n', 'iris_project/pipelines/iris_classifier/__init__.py': '', 'requirements.txt': 'mage-ai==0.9.80\nipykernel==6.15.0\npandas==2.3.3\nscikit-learn==1.7.2\npika==1.4.4\npytest==9.2.0\nsnowflake-connector-python==4.0.0\nnbformat==5.11.1\nnbclient==0.11.0\n'}
for name,text in FILES.items():
    path=ROOT/name;path.parent.mkdir(parents=True,exist_ok=True)
    if not path.exists(): path.write_text(text)
if COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
    subprocess.run([sys.executable,'-m','uv','venv','-p','3.12','/content/mage_env'],check=True)
    PYTHON=Path('/content/mage_env/bin/python')
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',str(PYTHON),'-r','requirements.txt'],check=True)
else:
    PYTHON=Path(sys.executable)
MAGE=PYTHON.with_name('mage')
print('Execution environment:', 'Google Colab' if COLAB else 'Local Python 3.12')
print('Project files ready:',len(FILES))


## C1: Actual Mage CLI and block tests

In [2]:
result=subprocess.run([str(MAGE),'run','iris_project','iris_classifier','--test'],capture_output=True,text=True,cwd=ROOT)
print(result.stdout); print(result.stderr)
assert result.returncode == 0

DBT library not installed.
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth = 3 | Test accuracy = 1.0
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix:
Saved 30 predictions to iris_predictions.csv
predicted   0  1   2
actual              
0          10  0   0
1           0  9   0
2           0  0  11
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.

INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run



## C2: Depth 1–5, deleting predictions before each run

In [3]:
result=subprocess.run([str(PYTHON),'iris_project/depth_experiment.py'],capture_output=True,text=True,cwd=ROOT)
print(result.stdout);print(result.stderr)
assert result.returncode==0
print((ROOT/'evidence/iris-depths.json').read_text())

[
  {
    "max_depth": 1,
    "accuracy": 0.633,
    "passed": false,
    "csv_created": false
  },
  {
    "max_depth": 2,
    "accuracy": 0.967,
    "passed": true,
    "csv_created": true
  },
  {
    "max_depth": 3,
    "accuracy": 1.0,
    "passed": true,
    "csv_created": true
  },
  {
    "max_depth": 4,
    "accuracy": 1.0,
    "passed": true,
    "csv_created": true
  },
  {
    "max_depth": 5,
    "accuracy": 1.0,
    "passed": true,
    "csv_created": true
  }
]
Restored default depth 3 successfully


[
  {
    "max_depth": 1,
    "accuracy": 0.633,
    "passed": false,
    "csv_created": false
  },
  {
    "max_depth": 2,
    "accuracy": 0.967,
    "passed": true,
    "csv_created": true
  },
  {
    "max_depth": 3,
    "accuracy": 1.0,
    "passed": true,
    "csv_created": true
  },
  {
    "max_depth": 4,
    "accuracy": 1.0,
    "passed": true,
    "csv_created": true
  },
  {
    "max_depth": 5,
    "accuracy": 1.0,
    "passed": true,
    "csv_created": true
  }
]



## C3: Evaluation branch

In [4]:
print((ROOT/'iris_project/pipelines/iris_classifier/metadata.yaml').read_text())
print((ROOT/'evidence/iris-default.txt').read_text())

blocks:
- name: load_iris
  uuid: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks: [verify_iris]
- name: verify_iris
  uuid: verify_iris
  type: transformer
  language: python
  upstream_blocks: [load_iris]
  downstream_blocks: [prepare_data]
- name: prepare_data
  uuid: prepare_data
  type: transformer
  language: python
  upstream_blocks: [verify_iris]
  downstream_blocks: [train_model]
- name: train_model
  uuid: train_model
  type: transformer
  language: python
  upstream_blocks: [prepare_data]
  downstream_blocks: [save_results, evaluate_model]
- name: save_results
  uuid: save_results
  type: data_exporter
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
- name: evaluate_model
  uuid: evaluate_model
  type: transformer
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
name: iris_classifier
uuid: iris_classifier
type: python
variables:
  max_depth: 3

DBT library not installed.
Rows lo

## C4: Added verify_iris transformer and its test

In [5]:
print((ROOT/'iris_project/transformers/verify_iris.py').read_text())

"""C4 extension: validate the input measurements before splitting."""
if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

@transformer
def verify(df, *args, **kwargs):
    return df.copy()

@test
def test_input_quality(output, *args):
    assert not output.isna().any().any(), 'Iris data contains missing values'
    assert set(output['target']) == {0, 1, 2}, 'Expected exactly three Iris classes'



## Temporary authenticated Mage UI through Cloudflare
Stop services and clear this cell output before sharing. This output contains a temporary preview secret.

In [ ]:
if COLAB:
    import time, re, secrets, urllib.request
    for process_name in ['tunnel', 'server']:
        process = globals().get(process_name)
        if process is not None:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill()
    download = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    urllib.request.urlretrieve(download, '/content/cloudflared')
    Path('/content/cloudflared').chmod(0o755)
    preview_password = secrets.token_urlsafe(32)
    preview_database = '/content/mage-preview-' + secrets.token_hex(8) + '.db'
    env = os.environ.copy()
    env.update(REQUIRE_USER_AUTHENTICATION='True',
               DEFAULT_OWNER_EMAIL='23bcs006@iiitdmj.ac.in',
               DEFAULT_OWNER_PASSWORD=preview_password,
               DEFAULT_OWNER_USERNAME='abhinav',
               MAGE_DATABASE_CONNECTION_URL='sqlite:///' + preview_database)
    server = subprocess.Popen([str(MAGE), 'start', 'iris_project', '--host', '127.0.0.1', '--port', '6789'],
                              cwd=ROOT, env=env, stdout=open('/content/secure-mage.log', 'w'), stderr=subprocess.STDOUT)
    for attempt in range(90):
        try:
            urllib.request.urlopen('http://127.0.0.1:6789', timeout=2)
            break
        except Exception: time.sleep(1)
    else: raise RuntimeError('Mage failed to start; inspect secure-mage.log privately')
    tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:6789', '--no-autoupdate'],
                              stdout=open('/content/secure-tunnel.log', 'w'), stderr=subprocess.STDOUT)
    urls = []
    for attempt in range(45):
        text = Path('/content/secure-tunnel.log').read_text()
        urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
        if urls: break
        time.sleep(1)
    print('Mage URL:', urls[0] if urls else 'Not ready')
    print('Preview username: 23bcs006@iiitdmj.ac.in')
    print('Temporary preview password:', preview_password)
    print('After the screenshot: stop services, then clear this cell output before saving or sharing.')
else:
    print('UI preview requires Google Colab; use mage start locally for a local preview.')
